# 05 - SQL Analysis with DuckDB

In this notebook, we will query the star schema CSV files using SQL.

DuckDB lets us run SQL directly inside Python/Jupyter without setting up a separate database server.

We will use SQL to answer questions like:

- What is the overall churn rate?
- How much monthly revenue is at risk?
- Which contract types have higher churn?
- Which payment methods have higher churn?
- Does the data support our assumption that electronic check is higher risk?

## 1. Import Libraries

`duckdb` runs SQL queries.

`pandas` displays the query results nicely in the notebook.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

## 2. Connect to DuckDB

This creates an in-memory DuckDB connection.

That means the database exists while this notebook is running. We are not creating a permanent database file yet.

In [2]:
con = duckdb.connect(database=":memory:")

con

## 3. Register the Star Schema CSV Files

We will read each CSV into a DuckDB table.

These CSVs were created in notebook 04.

In [3]:
schema_dir = Path("../data/processed/star_schema")

tables = {
    "dim_customer": schema_dir / "dim_customer.csv",
    "dim_contract": schema_dir / "dim_contract.csv",
    "dim_payment_method": schema_dir / "dim_payment_method.csv",
    "dim_service": schema_dir / "dim_service.csv",
    "dim_churn_reason": schema_dir / "dim_churn_reason.csv",
    "fact_customer_snapshot": schema_dir / "fact_customer_snapshot.csv",
}

for table_name, file_path in tables.items():
    con.execute(
        f"""
        create or replace table {table_name} as
        select *
        from read_csv_auto('{file_path.as_posix()}')
        """
    )

con.sql("show tables").df()

,name
0,dim_churn_reason
1,dim_contract
2,dim_customer
3,dim_payment_method
4,dim_service
5,fact_customer_snapshot


## 4. Validate Row Counts

Before analysis, check that the loaded SQL tables have the row counts we expect.

In [4]:
con.sql("""
select 'dim_customer' as table_name, count(*) as row_count from dim_customer
union all
select 'dim_contract', count(*) from dim_contract
union all
select 'dim_payment_method', count(*) from dim_payment_method
union all
select 'dim_service', count(*) from dim_service
union all
select 'dim_churn_reason', count(*) from dim_churn_reason
union all
select 'fact_customer_snapshot', count(*) from fact_customer_snapshot
order by table_name;
""").df()

,table_name,row_count
0,dim_churn_reason,21
1,dim_contract,6
2,dim_customer,7043
3,dim_payment_method,4
4,dim_service,322
5,fact_customer_snapshot,7043


**Your notes:**

- Do the SQL row counts match what you saw in notebook 04?
- Why is this validation step useful?

## 5. Overall Customer and Revenue Metrics

This query calculates the top-level numbers for the executive overview dashboard.

In [5]:
con.sql("""
select
    count(*) as total_customers,
    sum(churn_value) as churned_customers,
    round(sum(churn_value) * 1.0 / count(*), 4) as churn_rate,
    round(sum(monthly_charges), 2) as monthly_recurring_revenue,
    round(sum(revenue_at_risk), 2) as monthly_revenue_at_risk,
    round(avg(monthly_charges), 2) as average_monthly_charge,
    round(avg(cltv), 2) as average_cltv
from fact_customer_snapshot;
""").df()

,total_customers,churned_customers,churn_rate,monthly_recurring_revenue,monthly_revenue_at_risk,average_monthly_charge,average_cltv
0,7043,1869.0,0.2654,456116.6,139130.85,64.76,4400.3


SQL details:

- `count(*)` counts rows/customers.
- `sum(churn_value)` works because churned customers are `1` and non-churned customers are `0`.
- `sum(churn_value) / count(*)` gives the churn rate.
- `round(..., 2)` makes money-style numbers easier to read.

## 6. Churn by Contract Type

Now we join the fact table to `dim_contract`.

The fact table stores `contract_key`, and the dimension table tells us what that key means.

In [6]:
con.sql("""
select
    c.contract,
    c.paperless_billing,
    count(*) as total_customers,
    sum(f.churn_value) as churned_customers,
    round(sum(f.churn_value) * 1.0 / count(*), 4) as churn_rate,
    round(sum(f.monthly_charges), 2) as monthly_revenue,
    round(sum(f.revenue_at_risk), 2) as monthly_revenue_at_risk
from fact_customer_snapshot as f
join dim_contract as c
    on f.contract_key = c.contract_key
group by
    c.contract,
    c.paperless_billing
order by churn_rate desc;
""").df()

,contract,paperless_billing,total_customers,churned_customers,churn_rate,monthly_revenue,monthly_revenue_at_risk
0,Month-to-month,True,2586,1249.0,0.4830,188010.80,95936.45
1,Month-to-month,False,1289,406.0,0.3150,69283.35,24910.65
2,One year,True,800,118.0,0.1475,60650.60,10445.15
3,One year,False,673,48.0,0.0713,35166.00,3673.30
4,Two year,True,785,33.0,0.0420,58131.40,3128.40
5,Two year,False,910,15.0,0.0165,44874.45,1036.90


**Your notes:**

- Which contract group has the highest churn rate?
- Does month-to-month look riskier than one-year or two-year contracts?
- Why might contract length affect churn?

## 7. Churn by Payment Method

This query tests our earlier assumption about electronic check.

In [7]:
con.sql("""
select
    p.payment_method,
    p.payment_risk_classification,
    count(*) as total_customers,
    sum(f.churn_value) as churned_customers,
    round(sum(f.churn_value) * 1.0 / count(*), 4) as churn_rate,
    round(avg(f.monthly_charges), 2) as average_monthly_charge,
    round(sum(f.revenue_at_risk), 2) as monthly_revenue_at_risk
from fact_customer_snapshot as f
join dim_payment_method as p
    on f.payment_method_key = p.payment_method_key
group by
    p.payment_method,
    p.payment_risk_classification
order by churn_rate desc;
""").df()

,payment_method,payment_risk_classification,total_customers,churned_customers,churn_rate,average_monthly_charge,monthly_revenue_at_risk
0,Electronic check,Higher observed churn risk,2365,1071.0,0.4529,76.26,84288.75
1,Mailed check,Manual payment method,1612,308.0,0.1911,43.92,16803.60
2,Bank transfer (automatic),Automatic payment method,1544,258.0,0.1671,67.19,20091.90
3,Credit card (automatic),Automatic payment method,1522,232.0,0.1524,66.51,17946.60


**Your notes:**

- Which payment method has the highest churn rate?
- Does the SQL result support the electronic check risk assumption?
- What is the difference between an assumption and an insight supported by data?

## 8. Churn by Internet Service

This query uses the service dimension.

Instead of grouping by all service combinations, we start with just `internet_service` because it is easier to interpret.

In [8]:
con.sql("""
select
    s.internet_service,
    count(*) as total_customers,
    sum(f.churn_value) as churned_customers,
    round(sum(f.churn_value) * 1.0 / count(*), 4) as churn_rate,
    round(avg(f.monthly_charges), 2) as average_monthly_charge,
    round(sum(f.revenue_at_risk), 2) as monthly_revenue_at_risk
from fact_customer_snapshot as f
join dim_service as s
    on f.service_key = s.service_key
group by
    s.internet_service
order by churn_rate desc;
""").df()

,internet_service,total_customers,churned_customers,churn_rate,average_monthly_charge,monthly_revenue_at_risk
0,Fiber optic,3096,1297.0,0.4189,91.50,114300.05
1,DSL,2421,459.0,0.1896,58.10,22529.20
2,No,1526,113.0,0.0740,21.08,2301.60


## 9. Top Churn Reasons

Here we join to `dim_churn_reason`.

We filter out `Not churned` because we only want reasons from customers who actually left.

In [9]:
con.sql("""
select
    r.churn_reason,
    count(*) as churned_customers,
    round(sum(f.monthly_charges), 2) as monthly_revenue_at_risk,
    round(avg(f.monthly_charges), 2) as average_monthly_charge,
    round(avg(f.cltv), 2) as average_cltv
from fact_customer_snapshot as f
join dim_churn_reason as r
    on f.churn_reason_key = r.churn_reason_key
where f.churn_value = 1
  and r.churn_reason <> 'Not churned'
group by
    r.churn_reason
order by churned_customers desc,
         monthly_revenue_at_risk desc
limit 10;
""").df()

,churn_reason,churned_customers,monthly_revenue_at_risk,average_monthly_charge,average_cltv
0,Attitude of support person,192,13980.85,72.82,4143.05
1,Competitor offered higher download speeds,189,14144.60,74.84,4164.74
2,Competitor offered more data,162,12351.75,76.25,4215.91
3,Don't know,154,11099.05,72.07,4105.35
4,Competitor made better offer,140,10672.10,76.23,4204.21
5,Attitude of service provider,135,10399.30,77.03,4223.23
6,Competitor had better devices,130,9432.35,72.56,4130.72
7,Network reliability,103,7497.55,72.79,4077.47
8,Product dissatisfaction,102,7528.65,73.81,4208.71
9,Price too high,98,7398.55,75.50,4226.59


## 10. Highest Priority Customers

For retention operations, we need to identify customers the business might contact first.

This first version sorts by:

- churn score
- monthly charges
- CLTV

Later, we can create a more transparent priority score.

In [10]:
con.sql("""
select
    f.customerid,
    c.city,
    dc.contract,
    p.payment_method,
    f.tenure_months,
    f.monthly_charges,
    f.churn_score,
    f.cltv
from fact_customer_snapshot as f
join dim_customer as c
    on f.customerid = c.customerid
join dim_contract as dc
    on f.contract_key = dc.contract_key
join dim_payment_method as p
    on f.payment_method_key = p.payment_method_key
where f.churn_value = 0
order by
    f.churn_score desc,
    f.monthly_charges desc,
    f.cltv desc
limit 20;
""").df()

,customerid,city,contract,payment_method,tenure_months,monthly_charges,churn_score,cltv
0,5914-XRFQB,Caspar,Two year,Bank transfer (automatic),72,115.80,80,6479
1,3396-DKDEL,King City,Two year,Credit card (automatic),70,115.15,80,4388
2,0619-OLYUR,Garden Grove,Two year,Credit card (automatic),72,111.90,80,6437
3,2499-AJYUA,Benton,Two year,Credit card (automatic),72,110.80,80,5933
4,3766-EJLFL,Mill Creek,Two year,Bank transfer (automatic),68,109.05,80,6391
5,4016-BJKTZ,San Jose,Two year,Electronic check,25,108.90,80,4398
6,9137-UIYPG,Butte City,Month-to-month,Electronic check,35,106.90,80,5447
7,4010-YLMVT,Yucaipa,Month-to-month,Credit card (automatic),56,106.60,80,5056
8,8777-MBMTS,West Covina,Two year,Credit card (automatic),65,105.85,80,4872
9,5074-FBGHB,Freedom,One year,Credit card (automatic),64,104.65,80,6443


**Your notes:**

- Why do we filter to `churn_value = 0` here?
- Why might high churn score plus high monthly charges matter for retention work?
- Would you contact churned customers or non-churned high-risk customers first?

## 11. Save SQL Outputs for Documentation

These exports are optional, but useful for documentation or dashboard planning.

In [11]:
analysis_dir = Path("../data/processed/analysis_outputs")
analysis_dir.mkdir(parents=True, exist_ok=True)

payment_churn = con.sql("""
select
    p.payment_method,
    p.payment_risk_classification,
    count(*) as total_customers,
    sum(f.churn_value) as churned_customers,
    round(sum(f.churn_value) * 1.0 / count(*), 4) as churn_rate,
    round(sum(f.revenue_at_risk), 2) as monthly_revenue_at_risk
from fact_customer_snapshot as f
join dim_payment_method as p
    on f.payment_method_key = p.payment_method_key
group by
    p.payment_method,
    p.payment_risk_classification
order by churn_rate desc;
""").df()

payment_churn.to_csv(analysis_dir / "payment_method_churn.csv", index=False)

analysis_dir / "payment_method_churn.csv"

WindowsPath('../data/processed/analysis_outputs/payment_method_churn.csv')

## 12. Next Step

Next, we will define the key dashboard measures and create a simple DAX measure list for Power BI.